In [3]:
%pip install beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import requests
from bs4 import BeautifulSoup
from pathlib import Path

In [26]:
OUTPUT_DIR = Path.cwd().parent / "data" / "raw" / "web_scraped"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; BabyOS-research-bot/1.0; "
        "educational project)"
    )
}


In [5]:
def clean_text(soup: BeautifulSoup, selectors_to_remove: list[str]) -> str:
    """Remove nav/footer/ads and extract clean text."""
    for sel in selectors_to_remove:
        for tag in soup.select(sel):
            tag.decompose()
    return " ".join(soup.get_text(separator=" ").split())

In [27]:
def fetch_nhs_week_pages() -> None:
    """Fetch NHS pregnancy week-by-week pages."""
    print("\n--- Fetching NHS week-by-week pages ---")
    base = "https://www.nhs.uk/best-start-in-life/pregnancy/week-by-week-guide-to-pregnancy"
    
    # NHS week paths
    week_paths = {
        # 1st Trimester
        4:  "1st-trimester/week-4/",
        5:  "1st-trimester/week-5/",
        6:  "1st-trimester/week-6/",
        7:  "1st-trimester/week-7/",
        8:  "1st-trimester/week-8/",
        9:  "1st-trimester/week-9/",
        10: "1st-trimester/week-10/",
        11: "1st-trimester/week-11/",
        12: "1st-trimester/week-12/",
        13: "2nd-trimester/week-13/",
        14: "2nd-trimester/week-14/",
        16: "2nd-trimester/week-16/",
        18: "2nd-trimester/week-18/",
        20: "2nd-trimester/week-20/",
        22: "2nd-trimester/week-22/",
        24: "2nd-trimester/week-24/",
        26: "2nd-trimester/week-26/",
        28: "3rd-trimester/week-28/",
        30: "3rd-trimester/week-30/",
        32: "3rd-trimester/week-32/",
        34: "3rd-trimester/week-34/",
        36: "3rd-trimester/week-36/",
        38: "3rd-trimester/week-38/",
        40: "3rd-trimester/week-40/",
    }
    remove = ["nav", "footer", "header", ".breadcrumb", ".nhsuk-navigation",
              ".nhsuk-header", ".nhsuk-footer", "script", "style"]

    for week, path in week_paths.items():
        url = f"{base}/{path}"
        print(OUTPUT_DIR)
        out_file = OUTPUT_DIR / f"nhs_week_{week:02d}.txt"
        print(out_file)
        if out_file.exists():
            print(f"  Week {week}: already fetched, skipping.")
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            text = clean_text(soup, remove)
            out_file.write_text(
                f"SOURCE: NHS Pregnancy Week {week}\nURL: {url}\n\n{text}",
                encoding="utf-8"
            )
            print(f"  Week {week}: saved ({len(text)} chars)")
            time.sleep(1.5)  # polite crawl delay
        except Exception as e:
            print(f"  Week {week}: FAILED — {e}")

In [14]:
def fetch_nhs_topics() -> None:
    """Fetch key NHS pregnancy topic pages."""
    print("\n--- Fetching NHS pregnancy topic pages ---")
    topics = {
        "signs_of_labour":        "https://www.nhs.uk/pregnancy/labour-and-birth/signs-of-labour/",
        "antenatal_checks":       "https://www.nhs.uk/pregnancy/your-pregnancy-care/antenatal-checks/",
        "common_symptoms":        "https://www.nhs.uk/pregnancy/related-conditions/common-symptoms/",
        "foods_to_avoid":         "https://www.nhs.uk/pregnancy/keeping-well/foods-to-avoid/",
        "vitamins_supplements":   "https://www.nhs.uk/pregnancy/keeping-well/vitamins-supplements-and-nutrition/",
        "exercise":               "https://www.nhs.uk/pregnancy/keeping-well/exercise/",
        "mental_health":          "https://www.nhs.uk/pregnancy/keeping-well/mental-health/",
        "your_antenatal_team":    "https://www.nhs.uk/pregnancy/your-pregnancy-care/your-antenatal-care-team/",
        "gestational_diabetes":   "https://www.nhs.uk/conditions/gestational-diabetes/",
        "preeclampsia":           "https://www.nhs.uk/conditions/pre-eclampsia/",
        "miscarriage":            "https://www.nhs.uk/conditions/miscarriage/",
        "postnatal_depression":   "https://www.nhs.uk/mental-health/conditions/post-natal-depression/overview/",
        "breastfeeding":          "https://www.nhs.uk/conditions/baby/breastfeeding-and-bottle-feeding/breastfeeding/",
        "newborn_checks":         "https://www.nhs.uk/conditions/baby/newborn-screening/checks-and-tests/",
    }

    remove = ["nav", "footer", "header", "script", "style",
              ".nhsuk-navigation", ".nhsuk-header", ".nhsuk-footer",
              ".nhsuk-breadcrumb"]

    for name, url in topics.items():
        print(OUTPUT_DIR)
        out_file = OUTPUT_DIR / f"nhs_{name}.txt"
        if out_file.exists():
            print(f"  {name}: already fetched, skipping.")
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            text = clean_text(soup, remove)
            out_file.write_text(
                f"SOURCE: NHS — {name.replace('_', ' ').title()}\nURL: {url}\n\n{text}",
                encoding="utf-8"
            )
            print(f"  {name}: saved ({len(text)} chars)")
            time.sleep(1.5)
        except Exception as e:
            print(f"  {name}: FAILED — {e}")

In [8]:
def fetch_who_guidelines() -> None:
    """Fetch WHO antenatal care recommendations overview."""
    print("\n--- Fetching WHO guidelines ---")
    urls = {
        "who_antenatal_overview": "https://www.who.int/news-room/fact-sheets/detail/pregnancy",
        "who_maternal_health":    "https://www.who.int/health-topics/maternal-health",
    }
    remove = ["nav", "footer", "header", "script", "style"]

    for name, url in urls.items():
        out_file = OUTPUT_DIR / f"{name}.txt"
        if out_file.exists():
            print(f"  {name}: already fetched, skipping.")
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            text = clean_text(soup, remove)
            out_file.write_text(
                f"SOURCE: WHO — {name}\nURL: {url}\n\n{text}",
                encoding="utf-8"
            )
            print(f"  {name}: saved ({len(text)} chars)")
            time.sleep(2)
        except Exception as e:
            print(f"  {name}: FAILED — {e}")

In [9]:

def report() -> None:
    files = list(OUTPUT_DIR.glob("*.txt"))
    total_chars = sum(f.stat().st_size for f in files)
    print(f"\n✅ Corpus fetch complete.")
    print(f"   Files saved: {len(files)}")
    print(f"   Total size:  {total_chars / 1024:.1f} KB")
    print(f"   Location:    {OUTPUT_DIR}")

In [28]:
# if __name__ == "__main__":
fetch_nhs_week_pages()
# fetch_nhs_topics()
# fetch_who_guidelines()
# report()



--- Fetching NHS week-by-week pages ---
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_04.txt
  Week 4: saved (6593 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_05.txt
  Week 5: saved (7471 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_06.txt
  Week 6: saved (6818 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_07.txt
  Week 7: saved (7503 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_08.txt
  Week 8: saved (6971 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_09.txt
  Week 9: saved (7496 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
c:\Ironhack\W8\BabyOS\data\raw\web_scraped\nhs_week_10.txt
  Week 10: saved (7410 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scrape

In [29]:
fetch_nhs_topics()


--- Fetching NHS pregnancy topic pages ---
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  signs_of_labour: saved (6539 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  antenatal_checks: FAILED — 404 Client Error: Not Found for url: https://www.nhs.uk/pregnancy/your-pregnancy-care/antenatal-checks/
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  common_symptoms: saved (627 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  foods_to_avoid: saved (7939 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  vitamins_supplements: saved (7833 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  exercise: saved (6767 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  mental_health: saved (4169 chars)
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  your_antenatal_team: FAILED — 404 Client Error: Not Found for url: https://www.nhs.uk/pregnancy/your-pregnancy-care/your-antenatal-care-team/
c:\Ironhack\W8\BabyOS\data\raw\web_scraped
  gestational_diabetes: saved (7462 chars)
c:\Ironhack\W8\BabyOS\data\

In [30]:
fetch_who_guidelines()


--- Fetching WHO guidelines ---
  who_antenatal_overview: saved (27 chars)
  who_maternal_health: saved (8085 chars)


In [31]:
report()


✅ Corpus fetch complete.
   Files saved: 37
   Total size:  263.0 KB
   Location:    c:\Ironhack\W8\BabyOS\data\raw\web_scraped
